In [ ]:
import time
import jax
import jax.numpy as jnp
import pytreeclass as tc
from loguru import logger
import fdtdx

: 

In [ ]:
def irradiance(
    E0_0: float,
    freq: float,
    z: float,
    sigma: float,
    eps_r: float = 1.0,
    mu_r: float = 1.0,
):
    """
    Analytical calculation of time-averaged power-flux density ⟨S⟩·k̂ of an
    attenuating plane wave propagating in a lossy medium given by real
    permittivity, real permeability, and real conductivity.

    Parameters
    ----------
    E0_0 : float
        Complex electric-field amplitude at z = 0 (peak value, V/m).
    freq : float
        Frequency of plane wave
    z : float
        Distance(s) from the reference plane (m), measured **along k̂**.
    sigma : float
        Conductivity as used in the vector form of Ohm's law. [S/m].
    eps_r : float, optional
        Relative permittivity ε_r (default 1.0).
    mu_r : float, optional
        Relative permeability μ_r (default 1.0).

    Returns
    -------
    I : jax.Array
        Irradiance I(z) = |E₀(z)|² / (2 η)  (W m⁻²).
    """
    omega = 2.0 * jnp.pi * freq
    eps = fdtdx.constants.eps0 * eps_r
    mu = fdtdx.constants.mu0 * mu_r
    # eta = jnp.sqrt(mu / eps)
    eta = jnp.sqrt((1j * omega * mu) / (sigma + (1j * omega * eps)))
    ratio = sigma / (omega * eps)

    root = jnp.sqrt(1.0 + ratio**2)
    alpha = omega * jnp.sqrt(mu * eps / 2.0) * jnp.sqrt(root - 1.0)
    E0_z = E0_0 * jnp.exp(-alpha * z)  # field amplitude vs. z
    return jnp.square(jnp.abs(E0_z)) / (2.0 * eta)